In [23]:
import json

In [24]:
# TATQA, HybridQA, ConvFinQA
DATA_NAME="HybridQA"
SPLIT="test"
DPR_PATH=f"../benchmark_data/{DATA_NAME}/{DATA_NAME}_{SPLIT}.jsonl"
CORPUS_PATH=f"../benchmark_data/{DATA_NAME}/{DATA_NAME}_corpus.json"

In [25]:
# "18--mixtral-8x22b", "67--mixtral-8x22b", "71--mixtral-8x22b"
dpr_id_to_dpr = dict()
with open(DPR_PATH) as in_file:
    for line in in_file:
        data = json.loads(line)
        dpr_id_to_dpr[data['dpr_id']] = data

In [26]:
with open(CORPUS_PATH) as in_file:
    corpus_data = json.load(in_file)

In [27]:
table_id_to_metadata = dict()
for table_data in corpus_data:
    table_id = table_data['table']['uid']
    table_title = table_data['table']['title']
    table_headers = table_data['table']['header']
    table_id_to_metadata[table_id] = [table_id, table_title, table_headers]

In [28]:
dpr_id = "18--mixtral-8x22b"

In [29]:
dpr_text = dpr_id_to_dpr[dpr_id]['DPR']
gt_tables = dpr_id_to_dpr[dpr_id]['ground_truth']['table']
gt_tables_metadata = []
for gt_table in gt_tables:
    gt_tables_metadata.append(table_id_to_metadata[gt_table])

In [42]:
create_dpr_excel(dpr_text, gt_tables_metadata, output_filename=f'{dpr_id}.xlsx')

Excel file saved as: 18--mixtral-8x22b.xlsx


In [45]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

def create_dpr_excel(dpr_text, gt_tables_metadata, output_filename='dpr_ground_truth.xlsx'):
    """
    Create an Excel file with DPR text and ground truth tables.
    
    Args:
        dpr_text: The DPR text string
        gt_tables_metadata: List of [table_id, table_title, table_headers]
        output_filename: Name of the output Excel file
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "DPR Ground Truth"
    
    # Styling
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF")

    dv = DataValidation(type="list", formula1='"Yes,No"', allow_blank=False)
    ws.add_data_validation(dv)
    
    # Write DPR text
    ws.merge_cells('A1:C1')
    ws['A1'] = "DPR:"
    ws['A1'].font = Font(bold=True, size=12)

    ws.merge_cells('A2:C2')
    ws['A2'] = dpr_text
    ws['A2'].alignment = Alignment(wrap_text=True, vertical='top')
    ws.row_dimensions[2].height = 60  # Adjust height for DPR text

    # Add spacing
    current_row = 4
    
    # Write Ground Truth header
    ws[f'A{current_row}'] = "DPR Annotation:"
    ws[f'A{current_row}'].font = Font(bold=True, size=12)
    current_row += 1

    # Write column headers
    ws[f'A{current_row}'] = "Criteria"
    ws[f'B{current_row}'] = "Annotation"
    for col in ['A', 'B']:
        ws[f'{col}{current_row}'].fill = header_fill
        ws[f'{col}{current_row}'].font = header_font
    current_row += 1
    
    # Write column headers
    ws[f'A{current_row}'] = "Is the DPR high-quality and in the right level of abstraction?"
    ws[f'A{current_row}'].alignment = Alignment(wrap_text=True, vertical='top')
    dv.add(f'B{current_row}')
    current_row += 1

    ws[f'A{current_row}'] = "Is the DPR clear i.e. is coherent, unambiguous, and easy to understand?"
    ws[f'A{current_row}'].alignment = Alignment(wrap_text=True, vertical='top')
    dv.add(f'B{current_row}')
    # Add spacing
    current_row += 2
    
    # Write Ground Truth header
    ws[f'A{current_row}'] = "Ground Truth:"
    ws[f'A{current_row}'].font = Font(bold=True, size=12)
    current_row += 1
    
    # Write column headers
    ws[f'A{current_row}'] = "Table ID"
    ws[f'B{current_row}'] = "Table Title"
    ws[f'C{current_row}'] = "Headers"
    ws[f'D{current_row}'] = "Valid"
    
    for col in ['A', 'B', 'C', 'D']:
        ws[f'{col}{current_row}'].fill = header_fill
        ws[f'{col}{current_row}'].font = header_font
    
    current_row += 1
    
    # Write ground truth tables (one per row)
    for table_metadata in gt_tables_metadata:
        table_id, table_title, table_headers = table_metadata
        
        ws[f'A{current_row}'] = table_id
        ws[f'B{current_row}'] = table_title
        ws[f'C{current_row}'] = ', '.join(table_headers)  # Join headers as comma-separated
        
        current_row += 1
    
    # Adjust column widths
    ws.column_dimensions['A'].width = 40
    ws.column_dimensions['B'].width = 50
    ws.column_dimensions['C'].width = 60
    
    # Save the workbook
    wb.save(output_filename)
    print(f"Excel file saved as: {output_filename}")